# Service Team Routing
## Capacitated Vehicle Routing Problem with Time Windows (CVRPTW)

The ability of service providers to set service time windows allows for easier and more dependable coordination between the service provider and their customers, while increasing overall customer satisfaction.

In this scenario we have a number of service order locations with associated time windows and service times (time on-site to complete service). Each technician has an associated availability, ability to complete certain types of service, and a maximum number of service appointments per day.

### Problem Details:
- 8 Locations each with an associated demand
    - 1 Headquarters 
        - service type 1 demand: [0]
        - service type 2 demand: [1]
        - headquarters hours of operation: [5,20]
    - 7 Service Locations
        - service type 1 demand: [1, 1, 1, 0, 0, 0, 0]
        - service type 2 demand: [0, 0, 1, 1, 1, 1, 1]
        - service locations time windows: [[9,12],[9,12],[11,14],[13,16],[13,16],[13,16],[13,16]]
        - service location service times: [ 1, 1, 1.5, 0.5, 0.5, 0.5]

- 3 Delivery vehicles each with an associated capacity
    - 3 service technicians
        - capacity for service type 1: [2, 1, 0]
        - capacity for service type 2: [0, 1, 4]
        - technician availability [[9,17], [12,15], [9,17]]
        

### Environment Setup
First, let's check if we have a GPU available in system. This is only applicable if you are running server through this notebook. Else comment the next cell.

In [ ]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>
            
            <h4>If running in Google Colab:</h4>
            <ol>
              <li>Click on <b>Runtime → Change runtime type</b></li>
              <li>Set <b>Hardware accelerator</b> to <b>GPU</b></li>
              <li>Then click <b>Save</b> and <b>Runtime → Restart runtime</b>.</li>
            </ol>
            
            <h4>If running in Docker:</h4>
            <ol>
              <li>Ensure you have <b>NVIDIA Docker runtime</b> installed (<code>nvidia-docker2</code>)</li>
              <li>Run container with GPU support: <code>docker run --gpus all ...</code></li>
              <li>Or use: <code>docker run --runtime=nvidia ...</code> for older Docker versions</li>
              <li>Verify GPU access: <code>docker run --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi</code></li>
            </ol>
            
            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()

#### Install Dependencies

In [ ]:
# Enable this in case you are running this in google colab or such places where cuOpt is not yet installed
#!pip uninstall -y cuda-python cuda-bindings cuda-core
#!pip install --upgrade --only-binary=:all: cuopt-server-cu12
#!pip install --upgrade --only-binary=:all: cuopt-server-cu13


In [ ]:
#Install notebook dependencies
!pip install -q matplotlib scipy pandas numpy

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from cuopt import routing
from cuopt.grpc.routing import RoutingClient


#### Run cuOpt gRPC server in background

Please disable the next cell in case you are trying to test against a server running on a different machine.


In [ ]:
import os
import subprocess
import atexit
import psutil
import time
from cuopt.grpc.routing import RoutingClient, RoutingSolveError

host = "localhost"
port = 5001

def is_cuopt_grpc_server_running():
    """True when a process was started as the cuopt_grpc_server binary."""
    for proc in psutil.process_iter(["cmdline"]):
        try:
            cmdline = proc.info["cmdline"] or []
            if cmdline and os.path.basename(cmdline[0]) == "cuopt_grpc_server":
                return True
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return False

def wait_for_server_health(timeout=60):
    """Wait until a routing client can connect to cuopt_grpc_server."""
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            RoutingClient(host, port, tls=False)
            print("Server is healthy!")
            return True
        except (RoutingSolveError, OSError):
            pass

        print(
            "Waiting for server to be ready... "
            f"({int(time.time() - start_time)}s elapsed)"
        )
        time.sleep(1)

    print(f"Server health check timed out after {timeout} seconds")
    return False

# Only start the server if it is not already running
if not is_cuopt_grpc_server_running():
    server_process = subprocess.Popen(
        ["cuopt_grpc_server", "--port", str(port), "--workers", "1"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    def cleanup_server():
        try:
            server_process.terminate()
            server_process.wait(timeout=5)
            print("Terminated cuOpt gRPC server")
        except (subprocess.TimeoutExpired, ProcessLookupError):
            try:
                server_process.kill()
                print("Killed cuOpt gRPC server")
            except ProcessLookupError:
                pass

    atexit.register(cleanup_server)
    print("Started cuOpt gRPC server")
else:
    print("cuOpt gRPC server is already running")

if not wait_for_server_health():
    print("Warning: Server may not be fully ready")


#### Utility functions

In [ ]:
# Used to plot the Co-ordinates
def gen_plot(df):
    plt.figure(figsize=(11, 11))
    plt.scatter(
        df["xcord"][:1],
        df["ycord"][:1],
        label="Depot",
        color="Green",
        marker="o",
        s=100,
    )
    plt.scatter(
        df["xcord"][1::],
        df["ycord"][1::],
        label="Locations",
        color="Red",
        marker="o",
        s=100,
    )
    plt.xlabel("x - axis")
    plt.ylabel("y - axis")
    plt.title("Simplified Map")
    plt.legend()

    for i, label in enumerate(df.index.values):
        plt.annotate(
            label,
            (df["xcord"].iloc[i], df["ycord"].iloc[i]),
            fontproperties=fm.FontProperties(size=16),
        )
    return plt


# Used to plot arrows
def add_arrows(df, route, plt, color="green"):
    prev_cord = ()
    for i, label in enumerate(route):
        if i > 0:
            plt.annotate(
                "",
                xy=(df["xcord"].iloc[label], df["ycord"].iloc[label]),
                xytext=prev_cord,
                arrowprops=dict(
                    arrowstyle="simple, head_length=0.5, head_width=0.5, tail_width=0.15",  # noqa
                    connectionstyle="arc3",
                    color=color,
                    mutation_scale=20,
                    ec="black",
                ),
                label="vehicle-1",
            )
        prev_cord = df["xcord"].iloc[label], df["ycord"].iloc[label]

    return plt


def routes_by_vehicle(solution):
    routes = {}
    for truck_id, location in zip(solution["truck_id"], solution["locations"]):
        routes.setdefault(int(truck_id), []).append(int(location))
    return routes


# Prints vehicle routes
def show_vehicle_routes(solution, locations):
    for vehicle_id, route in routes_by_vehicle(solution).items():
        print("For vehicle -", vehicle_id, "route is: \n")
        path = "->".join(locations[stop] for stop in route)
        print(path + "\n\n")


# Map vehicle routes
def map_vehicle_routes(df, solution, colors):
    plot = gen_plot(df)
    for idx, (vehicle_id, route) in enumerate(routes_by_vehicle(solution).items()):
        plot = add_arrows(df, route, plot, color=colors[idx])
    return plot


Below we visualize the service locations with respect to the service company headquarters. The cost from all locations to all other locations (a cost matrix) will be required for optimization. For the purpose of this simple example we will omit the cost matrix calculation.

In [ ]:
location_names       = [ "Headquarters",     "A",    "B",    "C",    "D",    "E",    "F",    "G"  ]
location_coordinates = [     [4, 4],        [1, 3], [8, 1], [2, 1], [6, 7], [0, 2], [7, 6], [5, 3] ]
location_coordinates_df = pd.DataFrame(location_coordinates, columns=['xcord', 'ycord'], index=location_names)
gen_plot(location_coordinates_df).show()

### Cost Matrix

The cost matrix dictates the cost of travel between locations of interest.  The cost itself can be anything relevant to the user.  In this case we are constraining time window constraints.  When constraining time windows for locations or vehicles it is assumed (if only a single cost matrix is provided) that it represents time. 

Here is the cost(time) matrix corresponding to the locations above:

In [ ]:
time_matrix = [
 [0.00, 0.31, 0.50, 0.36, 0.36, 0.44, 0.36, 0.14],
 [0.31, 0.00, 0.72, 0.22, 0.64, 0.14, 0.67, 0.40],
 [0.50, 0.72, 0.00, 0.60, 0.63, 0.80, 0.51, 0.36],
 [0.36, 0.22, 0.60, 0.00, 0.72, 0.22, 0.70, 0.36],
 [0.36, 0.64, 0.63, 0.72, 0.00, 0.77, 0.14, 0.41],
 [0.44, 0.14, 0.80, 0.22, 0.77, 0.00, 0.80, 0.51],
 [0.36, 0.67, 0.51, 0.70, 0.14, 0.80, 0.00, 0.36],
 [0.14, 0.40, 0.36, 0.36, 0.41, 0.51, 0.36, 0.00]
]

# Create a dataframe of this matrix
time_matrix_df  = pd.DataFrame(time_matrix, 
                              index=location_coordinates_df.index, 
                              columns=location_coordinates_df.index)
time_matrix_df

### Service Locations

Setup the service location data

In [ ]:
# exclude head quarters from service location names
service_location_ids = [1, 2, 3, 4, 5, 6, 7]
service_location_names = [location_names[i] for i in service_location_ids]
service_location_data = {
    "service_location_names": service_location_names,
    "service_location_ids": service_location_ids,
    "service_type1_demand": [1, 1, 1, 0, 0, 0, 0],
    "service_type2_demand": [0, 0, 1, 1, 1, 1, 1],
    "location_earliest_time": [9, 9, 11, 13, 13, 13, 13],
    "location_latest_time": [12, 12, 14, 16, 16, 16,16],
    "required_service_time": [1, 1, 1, 0, 0, 0, 0]
}
service_location_data_df = pd.DataFrame(service_location_data).set_index('service_location_names')
service_location_data_df

### Vehicles

Setup vehicle/technician data

In [ ]:
n_vehicles = 3
vehicle_data = {
    "vehicle_ids": [i for i in range(n_vehicles)],
    "capacity_service_type1":[2, 1, 0],
    "capacity_service_type2":[0, 1, 4],
    "vehicle_availability_earliest":[9, 11, 9],
    "vehicle_availability_latest":[17, 15, 17]
}
vehicle_data_df = pd.DataFrame(vehicle_data).set_index('vehicle_ids')
vehicle_data_df

# Initialize the gRPC routing client and problem model


In [ ]:
n_locations = len(location_names)
n_orders = len(service_location_ids)
data_model = routing.DataModel(n_locations, n_vehicles, n_orders)
client = RoutingClient(host, port, tls=False)


### Set Cost Matrix

In [ ]:
cost_matrix = np.array(time_matrix, dtype=np.float32)
data_model.add_cost_matrix(cost_matrix)


### Set Fleet Data

In [ ]:
data_model.set_vehicle_locations(
    np.zeros(n_vehicles, dtype=np.int32),
    np.zeros(n_vehicles, dtype=np.int32),
)
data_model.set_vehicle_time_windows(
    np.array(vehicle_data["vehicle_availability_earliest"], dtype=np.int32),
    np.array(vehicle_data["vehicle_availability_latest"], dtype=np.int32),
)
data_model.add_capacity_dimension(
    "service_type1",
    np.array(service_location_data["service_type1_demand"], dtype=np.int32),
    np.array(vehicle_data["capacity_service_type1"], dtype=np.int32),
)
data_model.add_capacity_dimension(
    "service_type2",
    np.array(service_location_data["service_type2_demand"], dtype=np.int32),
    np.array(vehicle_data["capacity_service_type2"], dtype=np.int32),
)


### Set Task Data

In [ ]:
data_model.set_order_locations(np.array(service_location_ids, dtype=np.int32))
data_model.set_order_time_windows(
    np.array(service_location_data["location_earliest_time"], dtype=np.int32),
    np.array(service_location_data["location_latest_time"], dtype=np.int32),
)
data_model.set_order_service_times(
    np.array(service_location_data["required_service_time"], dtype=np.int32)
)


### Set Solver Configuration

In [ ]:
solver_settings = routing.SolverSettings()
solver_settings.set_time_limit(5)


### Get Optimized Routes

In [ ]:
# Solve the problem
solution = client.solve(data_model, solver_settings)

if solution["status"] == 0:
    print("Cost for the routing in distance: ", solution["total_objective_value"])
    print("Vehicle count to complete routing: ", solution["vehicle_count"])
    show_vehicle_routes(solution, location_names)
else:
    print("NVIDIA cuOpt Failed to find a solution with status : ", solution["status"])
    print(solution["error_message"])


**Notice** that this solution leverages the fact that vehicle 1 is the only vehicle with the ability to perform both service type 1 and service type 2.  In addition, vehicle 0 and vehicle 2 also serve the locations they are suited to service and minimize the time taken along these routes.

In [ ]:
vehicle_colors = ["red", "green", "blue"]
map_vehicle_routes(location_coordinates_df, solution, vehicle_colors).show()


## License

SPDX-FileCopyrightText: Copyright (c) 2025 NVIDIA CORPORATION & AFFILIATES. All rights reserved.

SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.